# Live Demo — AI Code Review (Module 2)

**For the presentation.** Paste a code snippet, click **Review this code**, and the
fine-tuned CodeT5+ model writes a review comment — live.

**How to run (takes ~1 minute to load):**
1. `Runtime → Run all` (a GPU is optional — this works on CPU).
2. Scroll to the box at the bottom, paste code, click the button.

No dataset download, no training — this loads the already-fine-tuned model and just
generates. The pre-filled examples below also run automatically as a safety net.


## 1. Setup


In [ ]:
# Pin transformers to 4.44.2 (newer breaks the CodeT5+ tokenizer). If this cell
# DOWNGRADES transformers, do Runtime -> Restart session, then Run all again.
!pip -q install "transformers==4.44.2"
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# --- Where to load the fine-tuned model from -------------------------------
# Option A (recommended): a Hugging Face Hub id (set this after Phase 2 upload).
HF_MODEL_ID = "your-username/codet5p-220m-msg-pyjs"
# Option B (fallback): a local/Drive path to exported weights. If set and it
# exists, it takes priority over HF_MODEL_ID.
LOCAL_WEIGHTS = ""   # e.g. "/content/drive/MyDrive/code-review-ai/weights/codet5p-220m-msg-pyjs"

import os
if LOCAL_WEIGHTS and os.path.exists(LOCAL_WEIGHTS):
    SRC = LOCAL_WEIGHTS
    if LOCAL_WEIGHTS.startswith("/content/drive"):
        from google.colab import drive; drive.mount("/content/drive")
else:
    SRC = HF_MODEL_ID

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model from: {SRC}  (device: {device})")
tokenizer = AutoTokenizer.from_pretrained(SRC)   # works on transformers 4.44.2
model = AutoModelForSeq2SeqLM.from_pretrained(SRC).to(device).eval()
print("Model ready.")


## 2. Review function


In [ ]:
MAX_SRC, MAX_TGT = 512, 128

def as_diff(code_text):
    # The model was trained on diff hunks (added lines start with '+'). Wrapping
    # pasted code as added lines matches that distribution and improves output.
    lines = code_text.strip("\n").splitlines()
    return "\n".join("+ " + ln for ln in lines)

@torch.no_grad()
def review(code_text, num_beams=5):
    diff = as_diff(code_text)
    enc = tokenizer(diff, truncation=True, max_length=MAX_SRC, return_tensors="pt").to(device)
    out = model.generate(**enc, max_new_tokens=MAX_TGT, num_beams=num_beams,
                         no_repeat_ngram_size=3)
    return tokenizer.decode(out[0], skip_special_tokens=True)


## 3. Pre-filled examples (run automatically)


In [ ]:
EXAMPLES = {
    "Python — SQL string built by concatenation": """
def get_user(conn, user_id):
    query = "SELECT * FROM users WHERE id = " + str(user_id)
    return conn.execute(query).fetchone()
""",
    "Python — bare except swallows errors": """
def load_config(path):
    try:
        return json.load(open(path))
    except:
        return {}
""",
    "JavaScript — == instead of === and no null check": """
function findItem(items, id) {
  for (var i = 0; i < items.length; i++) {
    if (items[i].id == id) return items[i];
  }
}
""",
}

for title, snippet in EXAMPLES.items():
    print("=" * 90)
    print("CODE:", title)
    print(snippet.strip())
    print("-> REVIEW:", review(snippet))
print("=" * 90)


## 4. Live: paste your own code and click the button


In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown

box = widgets.Textarea(
    value=EXAMPLES["Python — SQL string built by concatenation"].strip(),
    placeholder="Paste a Python or JavaScript snippet here...",
    layout=widgets.Layout(width="100%", height="180px"),
)
btn = widgets.Button(description="Review this code", button_style="primary",
                     icon="search")
out = widgets.Output()

def on_click(_):
    with out:
        out.clear_output()
        print("Reviewing...")
        comment = review(box.value)
        out.clear_output()
        display(Markdown(f"**AI review comment:**\n\n> {comment}"))

btn.on_click(on_click)
display(widgets.VBox([box, btn, out]))


---
*Model: fine-tuned `Salesforce/codet5p-220m` on the CodeReviewer comment-generation
data (Python + JavaScript). This is the Module-2 AI review layer; the full system
also adds a Semgrep security layer and posts fused comments on real GitHub PRs.*
